## Visualizing state reconstruction with a heterogeneous GridFM model

AC power flow is a **masked reconstruction** problem. For every bus in the grid some of
`(Vm, Va, Pg, Qg)` is measured and the rest has to be inferred from the network physics.
That framing is deliberately close to masked language modelling: a mask tells the model
which entries to fill in, and the grid topology plays the role the attention pattern
plays in a transformer.

This tutorial runs the whole loop end to end on the **IEEE 14-bus** test case:

1. train a small `GNS_heterogeneous` model on the case14 dataset shipped with the repo,
2. run it on one held-out load scenario,
3. plot the per-bus power balance residuals, and
4. compare ground truth / masked input / reconstruction for every bus quantity the
   model predicts.

The model here is intentionally tiny (4 layers, hidden size 32, 100 epochs on 58 load
scenarios) so the whole notebook finishes in well under a minute on a laptop CPU. It is
a visualization demo, **not** a training recipe — see the other configs in
`examples/config/` for realistic settings.

The grid is a PyG `HeteroData` graph with two node types, `bus` and `gen`, and three
edge types: `("bus", "connects", "bus")` for the transmission lines plus
`("gen", "connected_to", "bus")` / `("bus", "connected_to", "gen")` tying each generator
to its bus.

In [ ]:
import sys

if "google.colab" in sys.modules:
    try:
        !git clone https://github.com/gridfm/gridfm-graphkit.git
        %cd /content/gridfm-graphkit
        !pip install .
        %cd examples/notebooks/
    except Exception as e:
        print(f"Failed to start Google Collab setup, due to {e}")

In [ ]:
import tempfile

import numpy as np
import torch
import yaml
import lightning.pytorch as pl
from torch_geometric.loader import DataLoader

from gridfm_graphkit.datasets.globals import QG_H, QG_OUT, VA_H, VA_OUT, VM_H, VM_OUT
from gridfm_graphkit.datasets.hetero_powergrid_datamodule import LitGridHeteroDataModule
from gridfm_graphkit.io.param_handler import NestedNamespace, get_task
from gridfm_graphkit.utils.visualization import (
    visualize_error,
    visualize_quantity_heatmap,
)

## Load the YAML configuration

Everything in `gridfm-graphkit` is driven by a YAML config, which is wrapped in a
`NestedNamespace` so its values are read as attributes (`args.data.networks`,
`args.training.batch_size`). Component names in the config — the model type, the task,
the losses, the normalizer — are resolved through registries, so nothing here is
hard-coded to a particular architecture.

In [ ]:
CONFIG_PATH = "../config/HGNS_PF_case14_tutorial.yaml"
# The case14 dataset committed to the repo lives under `tests/data`; reuse it rather
# than shipping a second copy of the same parquet files.
DATA_PATH = "../../tests/data"

with open(CONFIG_PATH) as f:
    config_dict = yaml.safe_load(f)

args = NestedNamespace(**config_dict)
pl.seed_everything(args.seed, workers=True)

## Initialize the DataModule

`LitGridHeteroDataModule` reads the raw parquet files under
`<data_path>/<network>/raw/`, converts each load scenario into one `HeteroData` sample
in `processed/` (this happens once, on the first run), applies the task's masking
transform, and fits the normalizers on the training split.

For the `PowerFlow` task the mask follows the classical bus types (see
`AddPFHeteroMask` in `gridfm_graphkit/datasets/masking.py`):

| bus type | count in case14 | given to the model | masked → to be predicted |
| --- | --- | --- | --- |
| `PQ` (load) | 9 | `Pd`, `Qd` | `Vm`, `Va` |
| `PV` (generator) | 4 | `Vm`, and `Pg` on the generator node | `Va`, `Qg` |
| `REF` (slack) | 1 | `Va` = 0, the angle reference | `Vm`, `Qg`, and its generator's `Pg` |

One deviation from the textbook convention worth knowing before reading the plots: the
slack bus normally has its voltage *magnitude* specified, but here `Vm` is masked at the
reference bus as well — only the angle reference is given. So the slack appears among
the hidden buses below.

In [ ]:
data_module = LitGridHeteroDataModule(args, DATA_PATH)

## Build the task

`get_task` resolves `task.task_name` from the config to a Lightning module. The
normalizers are passed in so the task can de-normalize its predictions before computing
physics losses and metrics.

In [ ]:
task = get_task(args, data_module.data_normalizers)

## Train the model

100 epochs over 58 load scenarios takes roughly half a minute on a laptop CPU and is
enough to go from random output to a usable reconstruction on this small network.
Training 2.5x longer only improves masked-bus voltage magnitude MAE from 0.0071 to
0.0062 p.u., so this is a reasonable place to stop for a demo.

Logs and the normalizer statistics go to a temporary directory, so the notebook leaves
nothing behind in the repository.

In [ ]:
trainer = pl.Trainer(
    max_epochs=args.training.epochs,
    accelerator=args.training.accelerator,
    devices=args.training.devices,
    logger=pl.loggers.CSVLogger(save_dir=tempfile.mkdtemp()),
    enable_checkpointing=False,
    enable_model_summary=False,
    log_every_n_steps=1,
)
trainer.fit(task, datamodule=data_module)

## Run inference on a single held-out scenario

The plots below draw one grid, so the batch size has to be 1.

The model's raw head only outputs `(Vm, Va)`; the physics decoder inside
`GNS_heterogeneous` expands that to the 4-column `[Vm, Va, Pg, Qg]` bus layout that the
losses, metrics and plots all use. Note that predictions and targets use *different*
column layouts — hence the two families of index constants, `*_OUT` for
`output["bus"]` and `*_H` for `data["bus"].y` and `mask_dict["bus"]`.

In [ ]:
data_module.setup("test")
test_loader = DataLoader(data_module.test_datasets[0], batch_size=1, shuffle=False)
sample = next(iter(test_loader))

task.eval()
with torch.no_grad():
    output = task.model(sample)

# `HeteroDataMVANormalizer` divides every power quantity by a power base it fits on the
# training set (not the nominal 100 MVA of the case file), so multiplying by
# `sample.baseMVA` is what converts a normalized power back to MVA.
baseMVA = float(sample.baseMVA[0])
print(f"buses: {sample['bus'].x.shape[0]}, generators: {sample['gen'].x.shape[0]}")
print(f"bus predictions: {tuple(output['bus'].shape)}  (columns: Vm, Va, Pg, Qg)")
print(f"fitted power base: {baseMVA:.2f} MVA")

## How good is the reconstruction?

Before looking at any picture, put a number on it. The metric that matters is the error
on the **masked** buses only — the unmasked ones were handed to the model as inputs, so
reproducing them is not an achievement. A second, untrained copy of the same
architecture gives the scale of the problem to compare against.

Two things to expect in the output below. Voltages come out accurate: for the
`PowerFlow` task the reconstruction loss (`MaskedBusMSE`) supervises `Vm` and `Va`
against their targets. Reactive generation is not supervised that way — `Qg` falls out of
the model's physics decoder and is only constrained indirectly, through the power
balance residual term in the loss. Its error is correspondingly looser. That is a
property of the task setup, not a bug.

In [ ]:
untrained = get_task(args, data_module.data_normalizers)  # same model, random weights
untrained.eval()

quantities = [
    ("Voltage magnitude", VM_OUT, VM_H, "p.u.", 1.0),
    ("Voltage angle", VA_OUT, VA_H, "degrees", 180.0 / np.pi),
    ("Reactive Power Generated", QG_OUT, QG_H, "MVar", baseMVA),
]


def masked_mae(model, pred_col, target_col, scale):
    """Mean absolute error over the masked buses of the whole test split."""
    errors = []
    with torch.no_grad():
        for batch in test_loader:
            pred = model.model(batch)["bus"][:, pred_col]
            target = batch["bus"].y[:, target_col]
            mask = batch.mask_dict["bus"][:, target_col]
            errors.append((pred[mask] - target[mask]).abs() * scale)
    return torch.cat(errors).mean()


for name, pred_col, target_col, unit, scale in quantities:
    trained_mae = masked_mae(task, pred_col, target_col, scale)
    random_mae = masked_mae(untrained, pred_col, target_col, scale)
    print(
        f"{name:<26} MAE {trained_mae:>8.4f} {unit:<8}"
        f"(random init: {random_mae:.4f} {unit})",
    )

## Visualize nodal active power residuals

The power balance equations say that at every bus, generation minus demand must equal
the power flowing out over the connected lines. Feeding the model's predicted state into
those equations gives a per-bus **residual** in MW: zero means the prediction is
physically consistent at that bus, and a large value means it is not.

This is the model's self-consistency check, and it needs no ground truth — which is what
makes it useful at inference time on a grid state nobody has solved yet.

In all the plots below the node marker encodes the bus type: square = `REF` (slack),
hexagon = `PV` (generator), circle = `PQ` (load).

In [ ]:
residuals = visualize_error(sample, output, baseMVA=baseMVA)
print(f"worst bus residual: {residuals.abs().max():.3f} MW")

## Visualize the reconstruction of each predicted quantity

Three panels per quantity, on a shared colour scale:

1. **Ground truth** — the AC power flow solution.
2. **Masked** — what the model was actually given; grey nodes are the hidden values.
3. **Reconstructed** — the model's output, with the unmasked buses clamped back to
   ground truth so that only genuine reconstruction error is visible.

Panels 1 and 3 should be hard to tell apart. Where they differ, compare against the
grey nodes in panel 2: those are the buses the model had to infer.

The voltage plots are the ones to judge the model on. The reactive generation plot is
included to show the full bus output layout, and its looser agreement is the same
unsupervised-`Qg` effect quantified above.

In [ ]:
visualize_quantity_heatmap(
    sample,
    output,
    VM_OUT,
    VM_H,
    "Voltage magnitude",
    "p.u.",
)

In [ ]:
visualize_quantity_heatmap(
    sample,
    output,
    VA_OUT,
    VA_H,
    "Voltage angle",
    "degrees",
    scale=180.0 / np.pi,
)

In [ ]:
visualize_quantity_heatmap(
    sample,
    output,
    QG_OUT,
    QG_H,
    "Reactive Power Generated",
    "MVar",
    scale=baseMVA,
)